# المعمل الأول: بناء وتدريب شبكة عصبية اصطناعية (ANN)

**مادة:** الذكاء الاصطناعي والتعلم العميق
**البيئة:** `tf_env` — تأكد أن النواة (Kernel) المختارة أعلى اليمين هي **Python (TensorFlow)**

الهدف: بناء شبكة عصبية بسيطة تقوم بمهمة **تصنيف ثنائي (Binary Classification)**،
أي أن مخرجها إما `0` أو `1`.

> لتشغيل خلية: اضغط `Shift + Enter`

---
## 0) التحقق من البيئة

قبل أي شيء، نتأكد أن المكتبات مثبتة وأن النواة صحيحة.

In [ ]:
import sys
import numpy as np
import pandas as pd
import tensorflow as tf

print("Python     :", sys.version.split()[0])
print("NumPy      :", np.__version__)
print("Pandas     :", pd.__version__)
print("TensorFlow :", tf.__version__)

In [ ]:
# تثبيت العشوائية حتى نحصل جميعاً على نفس النتائج تقريباً
np.random.seed(42)
tf.random.set_seed(42)

---
## 1) تجهيز البيانات (Dataset)

ننشئ جدول بيانات افتراضي صغير:

- **المدخلات (Features):** عمودان `feature1` و `feature2`
- **المخرج (Label):** عمود واحد قيمته `0` أو `1` — وهو الإجابة الصحيحة التي تتعلم منها الشبكة

لاحظ النمط في البيانات: كلما زادت `feature1` ونقصت `feature2` مال التصنيف نحو `1`.

In [ ]:
data = {
    'feature1': [0.1, 0.2, 0.3, 0.4, 0.5],
    'feature2': [0.5, 0.4, 0.3, 0.2, 0.1],
    'label':    [0,   0,   1,   1,   1]
}

# تحويل البيانات إلى DataFrame
df = pd.DataFrame(data)
df

In [ ]:
# X = المدخلات (Features)
X = df[['feature1', 'feature2']].values

# y = الإجابات الصحيحة (Labels)
y = df['label'].values

print("شكل المدخلات X :", X.shape)   # (5 صفوف، 2 خاصية)
print("شكل المخرجات y :", y.shape)

---
## 2) بناء هيكل الشبكة (Sequential)

`Sequential` تعني أن الطبقات مرتبة بالتتابع، مخرج كل طبقة يدخل للتي بعدها.

| الطبقة | الوصف | دالة التفعيل |
|---|---|---|
| `Input(shape=(2,))` | طبقة المدخلات — تستقبل الخاصيتين | — |
| `Dense(8)` | الطبقة المخفية Hidden Layer — 8 عصبونات | `relu` |
| `Dense(1)` | طبقة المخرجات — عصبون واحد | `sigmoid` |

- **ReLU**: تُبقي القيم الموجبة وتُصفّر السالبة، وهي التي تمنح الشبكة قدرتها على تعلم العلاقات غير الخطية.
- **Sigmoid**: تضغط الناتج في المدى بين `0` و `1`، فيصبح قابلاً للقراءة كـ **احتمال**.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

model = Sequential([
    Input(shape=(2,)),               # لدينا 2 Features
    Dense(8, activation='relu'),     # Hidden Layer فيها 8 Neurons
    Dense(1, activation='sigmoid')   # Output Layer للتصنيف الثنائي
])

---
## 3) تجميع النموذج (Compile)

هنا نحدد **كيف تتعلم** الشبكة:

- `loss='binary_crossentropy'` — دالة قياس الخطأ المناسبة للتصنيف الثنائي.
- `optimizer='adam'` — الخوارزمية التي تُعدّل الأوزان لتقليل الخطأ.
- `metrics=['accuracy']` — مقياس نتابعه أثناء التدريب (نسبة الإجابات الصحيحة).

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

### عرض ملخص الشبكة

في العمود `Param #` سترى عدد الأوزان (Parameters) التي ستتعلمها الشبكة:

- الطبقة المخفية: `(2 مدخل × 8 عصبونات) + 8 انحيازات = 24`
- طبقة المخرجات: `(8 × 1) + 1 = 9`
- **المجموع = 33 معامل**

In [ ]:
model.summary()

---
## 4) تدريب النموذج (Fit)

- `epochs=100` — الشبكة تمر على كامل البيانات 100 مرة.
- `batch_size=1` — تُحدَّث الأوزان بعد كل صف على حدة.
- `verbose=1` — اطبع تفاصيل كل دورة.

راقب أثناء التشغيل: قيمة `loss` تنخفض تدريجياً، و `accuracy` ترتفع.

In [ ]:
history = model.fit(
    X,
    y,
    epochs=100,
    batch_size=1,
    verbose=1
)

### رسم منحنى التعلّم

الصورة أوضح من الأرقام: هذا الرسم يبيّن هل الشبكة تعلّمت فعلاً أم لا.

> **ما النتيجة المتوقعة؟** لا تتفاجأ إذا بقيت `loss` حول `0.6` ولم تصل `accuracy` إلى `1.0`.
> السبب أن بياناتنا **5 صفوف فقط**، وهذا لا يكفي إطلاقاً لتتعلم الشبكة بثقة.
> المهم هنا أن ترى المنحنى **ينزل تدريجياً** — هذا دليل أن آلية التعلّم تعمل.
> هذا ليس خطأ في الكود.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(history.history['loss'], color='crimson')
ax1.set_title('Loss (الخطأ)')
ax1.set_xlabel('Epoch')
ax1.grid(alpha=0.3)

ax2.plot(history.history['accuracy'], color='seagreen')
ax2.set_title('Accuracy (الدقة)')
ax2.set_xlabel('Epoch')
ax2.set_ylim(-0.05, 1.05)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5) اختبار النموذج

ندخل النقطة `feature1 = 0.2` و `feature2 = 0.4` ونرى ماذا يتوقع النموذج.

- الناتج `prediction` هو **احتمال** بين 0 و 1.
- نحوّله إلى تصنيف نهائي بقاعدة: إذا كان أكبر من `0.5` فهو `1`، وإلا فهو `0`.

> **انتبه لأمرين:**
>
> 1. هذه النقطة ليست جديدة فعلياً — إنها **نفس الصف الثاني في بيانات التدريب**
>    (`feature1=0.2, feature2=0.4`) وتصنيفها الحقيقي `0`.
>    لاختبار حقيقي على بيانات لم يرها النموذج استخدم قيمة غير موجودة في الجدول مثل `[[0.25, 0.35]]`.
> 2. **قد يخطئ النموذج ويعطيك `1`** رغم أن الإجابة الصحيحة `0`، وقد تختلف نتيجتك عن زميلك.
>    هذا متوقع تماماً مع 5 صفوف فقط: الاحتمال يخرج قريباً من `0.5` فيتأرجح التصنيف.
>    ولو أعدت تشغيل خلايا التدريب مرة أخرى فقد تحصل على نتيجة مختلفة.
>    **الدرس المستفاد:** كمية البيانات ليست تفصيلاً — هي أساس جودة النموذج.

In [ ]:
test_data = np.array([[0.2, 0.4]])

prediction = model.predict(test_data)

predicted_label = (prediction > 0.5).astype(int)

print("Prediction probability:", prediction)
print("Predicted label:", predicted_label[0][0])

---
## 6) تمارين إضافية (اختيارية)

جرّب التعديلات التالية وسجّل ما يحدث لـ `loss` و `accuracy`:

1. غيّر عدد العصبونات في الطبقة المخفية من `8` إلى `2` ثم إلى `64`.
2. قلّل `epochs` إلى `10` — هل تعلّمت الشبكة بما يكفي؟
3. أضف طبقة مخفية ثانية `Dense(8, activation='relu')`.
4. غيّر `optimizer` من `'adam'` إلى `'sgd'` — أيهما أسرع في خفض الخطأ؟
5. جرّب نقاط اختبار أخرى مثل `[[0.45, 0.15]]` و `[[0.05, 0.55]]`.

> **ملاحظة مهمة:** بياناتنا 5 صفوف فقط، وهذا عدد صغير جداً.
> الغرض منه فهم الآلية خطوة بخطوة، وليس بناء نموذج ذي قيمة عملية.